<a href="https://colab.research.google.com/github/bm606807-code/quiz-app/blob/master/website_content_fetcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4 python-docx

import requests
from bs4 import BeautifulSoup
from docx import Document
from urllib.parse import urljoin, urlparse

def get_all_website_content(base_url):
    visited = set()
    to_visit = {base_url}
    doc = Document()
    doc.add_heading(f'Content from {base_url}', 0)

    domain = urlparse(base_url).netloc

    while to_visit:
        current_url = to_visit.pop()
        if current_url in visited:
            continue

        try:
            print(f"Processing: {current_url}")
            response = requests.get(current_url, timeout=10)
            visited.add(current_url)

            if 'text/html' not in response.headers.get('Content-Type', ''):
                continue

            soup = BeautifulSoup(response.content, 'html.parser')

            # Add page title to Word doc
            title = soup.title.string if soup.title else current_url
            doc.add_heading(title, level=1)

            # Extract and add text paragraphs
            paragraphs = soup.find_all('p')
            for p in paragraphs:
                text = p.get_text().strip()
                if text:
                    doc.add_paragraph(text)

            # Find internal links to crawl
            for link in soup.find_all('a', href=True):
                full_url = urljoin(base_url, link['href'])
                # Ensure link is within the same domain and hasn't been visited
                if urlparse(full_url).netloc == domain and full_url not in visited:
                    to_visit.add(full_url)

        except Exception as e:
            print(f"Failed to crawl {current_url}: {e}")

    doc.save('website_content.docx')
    print("Finished! Content saved to website_content.docx")

# Run the scraper
get_all_website_content('https://www.tvsts.com/')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.7 MB/s eta 0:00:00
Processing: https://www.tvsts.com/
Processing: https://www.tvsts.com/news-and-updates/
Processing: https://www.tvsts.com/about/
Processing: https://www.tvsts.com/b2b/b2b-courses/
Processing: https://www.tvsts.com/lead/
Processing: https://www.tvsts.com/internship-programs/
Processing: https://www.tvsts.com/course/electronic-product-design-level-1-basic/
Processing: https://www.tvsts.com/course/tolerance-stack-up-analysis/
Processing: https://www.tvsts.com/course/statistical-process-control-spc/
Processing: https://www.tvsts.com/course/forklift-operator-training-certification/
Processing: https://www.tvsts.com/course/electric-vehicle/
Processing: https://www.tvsts.com/course/management-and-leadership-skills/
Processing: https://www.tvsts.com/#content
Processing: https://www.tvsts.com/course/bopt-crane-training-certification/
Processing: https://www.tvsts.com/course/cnc-machining-centre-basic-programming-mach

In [2]:
from google.colab import files
try:
    files.download('website_content.docx')
except FileNotFoundError:
    print('The file is still being generated or the crawler is still running. Please wait a moment.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>